In [ ]:
# ========== COMPLETE AI CHATBOT TRAINER WITH MULTI-MODEL SUPPORT ==========
import pandas as pd
import torch
import os
import io
import warnings
warnings.filterwarnings('ignore')

print("=" * 60)
print("AI CHATBOT TRAINER - MULTI-MODEL SUPPORT")
print("=" * 60)
print("Supported model types:")
print("  • Causal LM (decoder-only): DialoGPT, GPT-2, LLaMA, etc.")
print("  • Seq2Seq (encoder-decoder): T5, Flan-T5, BART, etc.")
print("=" * 60)

# Global variables
df = None
question_cols = []
answer_cols = []
model_name = None
model_type = None  # NEW: Track if 'causal' or 'seq2seq'
tokenizer = None
model = None
chatbot = None
save_path = "./my-fine-tuned-chatbot"
csv_filename = None
trainer = None
fine_tuned_model = None
fine_tuned_tokenizer = None
train_dataset = None
val_dataset = None
tokenized_train = None
tokenized_val = None

steps_completed = {
    'step1': False, 'step2': False, 'step3': False,
    'step4': False, 'step5': False, 'step6': False,
    'step7': False, 'step8': False, 'step9': False
}

# ========== MODEL TYPE DETECTION ==========
SEQ2SEQ_MODELS = ['t5', 'flan-t5', 'bart', 'mbart', 'marian', 'pegasus', 'blenderbot']
CAUSAL_MODELS = ['gpt2', 'dialogpt', 'gpt-j', 'gpt-neo', 'llama', 'phi', 'mistral', 'opt']

def detect_model_type(model_name):
    """Detect if model is causal LM or seq2seq"""
    name_lower = model_name.lower()
    for seq2seq in SEQ2SEQ_MODELS:
        if seq2seq in name_lower:
            return 'seq2seq'
    for causal in CAUSAL_MODELS:
        if causal in name_lower:
            return 'causal'
    # Default to causal
    return 'causal'

# ========== HELPER FUNCTIONS ==========
def parse_column_input(input_str, max_cols):
    if not input_str or input_str.strip() == "":
        return []
    input_str = input_str.strip().replace(' ', ',')
    indices = []
    if '-' in input_str and ',' not in input_str:
        try:
            start, end = map(int, input_str.split('-'))
            if 1 <= start <= max_cols and 1 <= end <= max_cols and start <= end:
                indices = list(range(start, end + 1))
            else:
                return None
        except:
            return None
    else:
        parts = input_str.replace(',', ' ').split()
        for part in parts:
            try:
                idx = int(part)
                if 1 <= idx <= max_cols:
                    indices.append(idx)
                else:
                    return None
            except:
                return None
    return sorted(list(set(indices)))

def display_column_selection(selected_cols, column_names, col_type):
    if selected_cols:
        print(f"\nCurrent {col_type} columns selected:")
        for idx in selected_cols:
            print(f"   Column {idx}: {column_names[idx-1]}")
    else:
        print(f"\nNo {col_type} columns selected yet.")

# ========== STEP 1: INSTALL LIBRARIES ==========
def step1_install_libraries():
    global steps_completed
    print("\n" + "="*60)
    print("STEP 1: INSTALLING REQUIRED LIBRARIES")
    print("="*60)
    !pip install -q torch transformers datasets accelerate pandas numpy scikit-learn evaluate rouge-score sentencepiece
    print("Libraries installed successfully!")
    steps_completed['step1'] = True
    return True

# ========== STEP 2: UPLOAD TRAINING CSV ==========
def step2_upload_csv():
    global df, question_cols, answer_cols, csv_filename, steps_completed
    print("\n" + "="*60)
    print("STEP 2: UPLOAD TRAINING CSV")
    print("="*60)
    question_cols = []
    answer_cols = []
    print("\nEnter the path to your training CSV or type 'upload':")
    file_input = input("Path: ").strip()
    if file_input.lower() == 'upload':
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            return False
        csv_filename = list(uploaded.keys())[0]
        df = pd.read_csv(io.BytesIO(uploaded[csv_filename]), on_bad_lines='skip', engine='python')
    else:
        csv_filename = file_input
        if not os.path.exists(csv_filename):
            print(f"File not found: {csv_filename}")
            return False
        df = pd.read_csv(csv_filename, on_bad_lines='skip')
    print(f"Loaded: {csv_filename}")
    print(f"Rows: {len(df)}, Columns: {len(df.columns)}")
    print("\nColumns:")
    for i, col in enumerate(df.columns):
        print(f"   {i+1}. {col}")

    # Select question columns
    while True:
        display_column_selection(question_cols, df.columns, "QUESTION")
        q_input = input("\nSelect QUESTION columns (numbers, 'done', or 'clear'): ").strip().lower()
        if q_input == 'done':
            if not question_cols:
                print("Must select at least one column.")
                continue
            break
        elif q_input == 'clear':
            question_cols.clear()
        elif q_input:
            new = parse_column_input(q_input, len(df.columns))
            if new:
                question_cols.extend(new)
                question_cols = sorted(list(set(question_cols)))

    # Select answer columns
    while True:
        display_column_selection(answer_cols, df.columns, "ANSWER")
        a_input = input("\nSelect ANSWER columns (numbers, 'done', or 'clear'): ").strip().lower()
        if a_input == 'done':
            if not answer_cols:
                print("Must select at least one column.")
                continue
            break
        elif a_input == 'clear':
            answer_cols.clear()
        elif a_input:
            new = parse_column_input(a_input, len(df.columns))
            if new:
                answer_cols.extend(new)
                answer_cols = sorted(list(set(answer_cols)))

    q_names = [df.columns[i-1] for i in question_cols]
    a_names = [df.columns[i-1] for i in answer_cols]
    print(f"\nFinal selection:")
    print(f"   Questions: {q_names}")
    print(f"   Answers: {a_names}")
    steps_completed['step2'] = True
    return True

# ========== STEP 3: PREPARE DATA ==========
def step3_prepare_data():
    global df, question_cols, answer_cols, train_dataset, val_dataset, model_type, steps_completed
    print("\n" + "="*60)
    print("STEP 3: PREPARING DATA FOR TRAINING")
    print("="*60)
    if df is None:
        print("No dataset loaded.")
        return False

    from sklearn.model_selection import train_test_split
    from datasets import Dataset

    q_names = [df.columns[i-1] for i in question_cols]
    a_names = [df.columns[i-1] for i in answer_cols]

    conversations = []
    for _, row in df.iterrows():
        q_parts = [str(row[col]).strip() for col in q_names if pd.notna(row[col]) and str(row[col]).strip()]
        a_parts = [str(row[col]).strip() for col in a_names if pd.notna(row[col]) and str(row[col]).strip()]
        if q_parts and a_parts:
            conversations.append({
                'question': " ".join(q_parts),
                'answer': " ".join(a_parts)
            })

    if not conversations:
        print("No valid conversations created.")
        return False

    # Format based on model type
    if model_type == 'seq2seq':
        # T5 format: "question: ... answer: ..."
        for conv in conversations:
            conv['text'] = f"question: {conv['question']} answer: {conv['answer']}"
    else:
        # DialoGPT/GPT format: "Human: ... Assistant: ..."
        for conv in conversations:
            conv['text'] = f"Human: {conv['question']}\nAssistant: {conv['answer']}"

    data_df = pd.DataFrame(conversations)
    train_df, val_df = train_test_split(data_df, test_size=0.2, random_state=42)
    train_dataset = Dataset.from_pandas(train_df)
    val_dataset = Dataset.from_pandas(val_df)

    print(f"Training samples: {len(train_df)}")
    print(f"Validation samples: {len(val_df)}")
    print(f"Format: {'Seq2Seq (T5 style)' if model_type == 'seq2seq' else 'Causal (GPT style)'}")
    print(f"\nSample: {conversations[0]['text'][:150]}...")
    steps_completed['step3'] = True
    return True

# ========== STEP 4: CHOOSE MODEL ==========
def step4_choose_model():
    global model_name, model_type, steps_completed
    print("\n" + "="*60)
    print("STEP 4: CHOOSE PRE-TRAINED MODEL")
    print("="*60)
    print("CAUSAL LM (Decoder-only):")
    print("1. microsoft/DialoGPT-small (Fast, 117M params)")
    print("2. gpt2 (Base GPT-2, 124M params)")
    print("\nSEQ2SEQ (Encoder-Decoder):")
    print("3. google/flan-t5-base (Instruction following, 250M params)")
    print("4. google/flan-t5-small (Faster T5, 80M params)")
    print("\nLARGER MODELS (Need more RAM):")
    print("5. microsoft/phi-2 (2.7B params)")
    print("6. google/flan-t5-large (780M params)")
    print("-" * 60)

    choice = input("Enter choice (1-6, default=1): ").strip() or "1"

    models = {
        "1": ("microsoft/DialoGPT-small", "causal"),
        "2": ("gpt2", "causal"),
        "3": ("google/flan-t5-base", "seq2seq"),
        "4": ("google/flan-t5-small", "seq2seq"),
        "5": ("microsoft/phi-2", "causal"),
        "6": ("google/flan-t5-large", "seq2seq"),
    }

    model_name, model_type = models.get(choice, models["1"])
    print(f"\nSelected: {model_name}")
    print(f"Model type: {model_type.upper()}")
    steps_completed['step4'] = True
    return True

# ========== STEP 5: SETUP MODEL AND TOKENIZER ==========
def step5_setup_model():
    global model_name, model_type, tokenizer, model, train_dataset, val_dataset, tokenized_train, tokenized_val, steps_completed
    print("\n" + "="*60)
    print("STEP 5: SETUP MODEL AND TOKENIZER")
    print("="*60)

    if model_name is None or train_dataset is None:
        print("Complete Steps 3 and 4 first.")
        return False

    from transformers import AutoTokenizer

    print(f"Loading tokenizer for {model_name}...")
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    # Handle pad token
    if tokenizer.pad_token is None:
        if model_type == 'seq2seq':
            tokenizer.pad_token = tokenizer.eos_token
        else:
            # For causal models, use last vocab token as pad
            tokenizer.pad_token_id = tokenizer.vocab_size - 1
            tokenizer.pad_token = tokenizer.convert_ids_to_tokens(tokenizer.pad_token_id)

    print(f"   Pad token: {tokenizer.pad_token} (ID: {tokenizer.pad_token_id})")

    # Load appropriate model class
    from transformers import AutoModelForCausalLM, AutoModelForSeq2SeqLM

    print(f"Loading {model_type.upper()} model...")
    use_cuda = torch.cuda.is_available()
    device = "cuda" if use_cuda else "cpu"

    if model_type == 'seq2seq':
        model = AutoModelForSeq2SeqLM.from_pretrained(model_name, torch_dtype=torch.float32)
    else:
        model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype=torch.float32)

    model = model.to(device)
    print(f"   Model loaded on: {device}")
    print(f"   Model dtype: {model.dtype}")

    # Tokenize based on model type
    print("Tokenizing data...")

    if model_type == 'seq2seq':
        # For T5: separate input and target
        def tokenize_seq2seq(examples):
            inputs = [text.split(" answer: ")[0] for text in examples["text"]]
            targets = [text.split(" answer: ")[1] if " answer: " in text else "" for text in examples["text"]]

            model_inputs = tokenizer(
                inputs,
                max_length=256,
                truncation=True,
                padding="max_length",
            )
            labels = tokenizer(
                targets,
                max_length=256,
                truncation=True,
                padding="max_length",
            )
            model_inputs["labels"] = labels["input_ids"]
            # Replace pad token ids with -100 for loss calculation
            model_inputs["labels"] = [
                [(l if l != tokenizer.pad_token_id else -100) for l in label]
                for label in model_inputs["labels"]
            ]
            return model_inputs

        tokenized_train = train_dataset.map(tokenize_seq2seq, batched=True, remove_columns=["text"])
        tokenized_val = val_dataset.map(tokenize_seq2seq, batched=True, remove_columns=["text"])
    else:
        # For causal LM: input = labels (shifted internally)
        def tokenize_causal(examples):
            result = tokenizer(
                examples["text"],
                truncation=True,
                padding="max_length",
                max_length=256,
                return_attention_mask=True
            )
            result["labels"] = result["input_ids"].copy()
            result["labels"] = [
                [(token if token != tokenizer.pad_token_id else -100) for token in labels]
                for labels in result["labels"]
            ]
            return result

        tokenized_train = train_dataset.map(tokenize_causal, batched=True, remove_columns=["text"])
        tokenized_val = val_dataset.map(tokenize_causal, batched=True, remove_columns=["text"])

    print(f"Tokenization complete!")
    print(f"   Train: {len(tokenized_train)}, Val: {len(tokenized_val)}")
    steps_completed['step5'] = True
    return True

# ========== STEP 6: CONFIGURE TRAINING ==========
def step6_configure_training():
    global trainer, tokenized_train, tokenized_val, model, tokenizer, steps_completed
    print("\n" + "="*60)
    print("STEP 6: CONFIGURE TRAINING")
    print("="*60)

    from transformers import TrainingArguments, Trainer

    use_cuda = torch.cuda.is_available()

    training_config = {
        "output_dir": "./my-chatbot",
        "num_train_epochs": 3,
        "warmup_steps": 50,
        "weight_decay": 0.01,
        "logging_dir": "./logs",
        "logging_steps": 10,
        "eval_strategy": "steps",
        "eval_steps": 50,
        "save_strategy": "steps",
        "save_steps": 100,
        "load_best_model_at_end": True,
        "report_to": "none",
        "save_total_limit": 2,
        "remove_unused_columns": False,
        "fp16": False,
        "gradient_accumulation_steps": 2 if use_cuda else 4,
        "per_device_train_batch_size": 4 if use_cuda else 2,
        "per_device_eval_batch_size": 4 if use_cuda else 2,
    }

    training_args = TrainingArguments(**training_config)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
    )

    print(f"Epochs: {training_args.num_train_epochs}")
    print(f"Batch size: {training_args.per_device_train_batch_size}")
    print(f"Device: {'GPU' if use_cuda else 'CPU'}")
    print(f"Model type: {model_type}")
    steps_completed['step6'] = True
    return True

# ========== STEP 7: START TRAINING ==========
def step7_start_training():
    global trainer, save_path, tokenizer, steps_completed
    print("\n" + "="*60)
    print("STEP 7: START TRAINING")
    print("="*60)

    if trainer is None:
        print("Configure training first.")
        return False

    confirm = input("Start training? (yes/no): ").strip().lower()
    if confirm not in ['yes', 'y']:
        print("Cancelled.")
        return False

    try:
        trainer.train()
        trainer.save_model(save_path)
        tokenizer.save_pretrained(save_path)
        print(f"\nTraining complete! Saved to: {save_path}")
        steps_completed['step7'] = True
        return True
    except Exception as e:
        print(f"Training failed: {e}")
        return False

# ========== STEP 8: LOAD FINE-TUNED MODEL ==========
def step8_load_model():
    global fine_tuned_model, fine_tuned_tokenizer, model_type, save_path, steps_completed
    print("\n" + "="*60)
    print("STEP 8: LOAD FINE-TUNED MODEL")
    print("="*60)

    if not os.path.exists(save_path):
        print("No trained model found. Train first.")
        return False

    from transformers import AutoTokenizer, AutoModelForCausalLM, AutoModelForSeq2SeqLM

    fine_tuned_tokenizer = AutoTokenizer.from_pretrained(save_path)
    if fine_tuned_tokenizer.pad_token is None:
        fine_tuned_tokenizer.pad_token = fine_tuned_tokenizer.eos_token

    print(f"Loading {model_type.upper()} model...")
    if model_type == 'seq2seq':
        fine_tuned_model = AutoModelForSeq2SeqLM.from_pretrained(save_path, torch_dtype=torch.float32)
    else:
        fine_tuned_model = AutoModelForCausalLM.from_pretrained(save_path, torch_dtype=torch.float32)

    if torch.cuda.is_available():
        fine_tuned_model = fine_tuned_model.to("cuda")

    print(f"Model loaded on: {fine_tuned_model.device}")
    print(f"Model dtype: {fine_tuned_model.dtype}")
    steps_completed['step8'] = True
    return True

# ========== STEP 9: CREATE CHATBOT ==========
def step9_create_chatbot():
    global chatbot, fine_tuned_model, fine_tuned_tokenizer, model_type, steps_completed
    print("\n" + "="*60)
    print("STEP 9: CREATE CHATBOT")
    print("="*60)

    if fine_tuned_model is None or fine_tuned_tokenizer is None:
        print("Load model first.")
        return False

    class Chatbot:
        def __init__(self, model, tokenizer, model_type):
            self.model = model
            self.tokenizer = tokenizer
            self.model_type = model_type
            self.model.eval()
            self.history = []
            print(f"   Chatbot type: {model_type.upper()}")

        def generate_response(self, user_input, max_length=200):
            try:
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()

                if self.model_type == 'seq2seq':
                    return self._generate_seq2seq(user_input, max_length)
                else:
                    return self._generate_causal(user_input, max_length)
            except Exception as e:
                print(f"\nError: {e}")
                return "I encountered an error. Please try again."

        def _generate_seq2seq(self, user_input, max_length):
            """T5/Flan-T5 generation"""
            # Build input with history context
            if self.history:
                context = " ".join([f"Q: {q} A: {a}" for q, a in self.history[-2:]])
                input_text = f"{context} Q: {user_input}"
            else:
                input_text = f"question: {user_input}"

            inputs = self.tokenizer(
                input_text,
                return_tensors="pt",
                truncation=True,
                max_length=256,
                padding=False
            )

            input_ids = inputs['input_ids'].to(self.model.device)
            attention_mask = inputs['attention_mask'].to(self.model.device)

            with torch.no_grad():
                outputs = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=max_length,
                    min_new_tokens=1,
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    top_p=0.9,
                    no_repeat_ngram_size=3,
                    repetition_penalty=1.2,
                )

            response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)

            # Clean up - remove input if echoed back
            if input_text in response:
                response = response.replace(input_text, "").strip()

            if not response:
                response = "I'm not sure how to respond."

            # Remove any "question:" prefix if present
            if response.lower().startswith("question:"):
                response = response[9:].strip()

            self.history.append((user_input, response))
            if len(self.history) > 6:
                self.history = self.history[-6:]

            return response

        def _generate_causal(self, user_input, max_length):
            """DialoGPT/GPT-2 generation"""
            if self.history:
                context = "\n".join([f"Human: {q}\nAssistant: {a}" for q, a in self.history[-2:]])
                prompt = f"{context}\nHuman: {user_input}\nAssistant:"
            else:
                prompt = f"Human: {user_input}\nAssistant:"

            inputs = self.tokenizer(
                prompt,
                return_tensors="pt",
                truncation=True,
                max_length=256,
                padding=False
            )

            input_ids = inputs['input_ids'].to(self.model.device)
            attention_mask = torch.ones_like(input_ids)

            # Check for invalid tokens
            vocab_size = self.model.config.vocab_size
            if input_ids.max() >= vocab_size:
                input_ids = torch.clamp(input_ids, 0, vocab_size - 1)

            with torch.no_grad():
                outputs = self.model.generate(
                    input_ids=input_ids,
                    attention_mask=attention_mask,
                    max_new_tokens=max_length,
                    min_new_tokens=1,
                    temperature=0.8,
                    do_sample=True,
                    pad_token_id=self.tokenizer.pad_token_id,
                    eos_token_id=self.tokenizer.eos_token_id,
                    top_p=0.9,
                    no_repeat_ngram_size=3,
                    repetition_penalty=1.2,
                )

            # Get only new tokens
            input_length = input_ids.shape[1]
            output_ids = outputs[0][input_length:]
            response = self.tokenizer.decode(output_ids, skip_special_tokens=True)

            if "Human:" in response:
                response = response.split("Human:")[0].strip()

            if not response:
                response = "I'm not sure how to respond."

            self.history.append((user_input, response))
            if len(self.history) > 6:
                self.history = self.history[-6:]

            return response

        def chat(self):
            print("\n" + "="*60)
            print(f"CHATBOT READY! ({self.model_type.upper()})")
            print(f"Device: {self.model.device}")
            print("="*60)
            print("Commands: 'quit', 'clear', 'test'")
            print("="*60)

            while True:
                try:
                    user_input = input("\nYou: ").strip()
                    if user_input.lower() in ['quit', 'exit', 'bye']:
                        print("\nChatbot: Goodbye!")
                        break
                    elif user_input.lower() == 'clear':
                        self.history = []
                        print("\nChatbot: History cleared!")
                        continue
                    elif user_input.lower() == 'test':
                        q = input("Test question: ").strip()
                        if q:
                            print("Chatbot:", self.generate_response(q))
                        continue
                    elif not user_input:
                        continue

                    print("Chatbot:", end=" ", flush=True)
                    print(self.generate_response(user_input))
                except KeyboardInterrupt:
                    print("\n\nChatbot: Goodbye!")
                    break

    try:
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        fine_tuned_model.eval()
        chatbot = Chatbot(fine_tuned_model, fine_tuned_tokenizer, model_type)
        print("   Chatbot created successfully!")
        steps_completed['step9'] = True
        return True
    except Exception as e:
        print(f"Error: {e}")
        return False

# ========== STEP 10-13: Same as before ==========
def step10_test_with_csv():
    global chatbot
    if chatbot is None:
        print("Create chatbot first.")
        return False
    from google.colab import files
    print("Upload test CSV...")
    uploaded = files.upload()
    if not uploaded:
        return True
    test_df = pd.read_csv(io.BytesIO(list(uploaded.values())[0]))
    print(f"\nColumns: {list(test_df.columns)}")
    q_col = input("Enter question column number: ").strip()
    try:
        q_col_name = test_df.columns[int(q_col)-1]
        for idx, row in test_df.head(10).iterrows():
            q = str(row[q_col_name])
            print(f"\nQ: {q[:80]}...")
            print(f"A: {chatbot.generate_response(q)[:100]}...")
    except Exception as e:
        print(f"Error: {e}")
    return True

def step11_start_chatting():
    global chatbot
    if chatbot is None:
        print("Create chatbot first.")
        return False
    chatbot.chat()
    return True

def step12_save_to_drive():
    global save_path
    if not os.path.exists(save_path):
        print("No model to save.")
        return False
    from google.colab import drive
    import shutil, datetime
    drive.mount('/content/drive')
    ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    dst = f"/content/drive/MyDrive/chatbot-{ts}"
    shutil.copytree(save_path, dst, dirs_exist_ok=True)
    print(f"Saved to: {dst}")
    return True

def step13_test_accuracy():
    global chatbot, val_dataset
    if chatbot is None:
        print("Create chatbot first.")
        return False
    print("Manual testing mode. Type 'done' to finish.")
    while True:
        q = input("\nQuestion: ").strip()
        if q.lower() == 'done':
            break
        if q:
            print(f"Response: {chatbot.generate_response(q)}")
    return True

# ========== RUN ESSENTIAL STEPS ==========
def run_essential_steps():
    print("\n" + "="*60)
    print("RUNNING ESSENTIAL STEPS")
    print("="*60)

    steps = [
        ("Install libraries", step1_install_libraries, 'step1'),
        ("Upload CSV", step2_upload_csv, 'step2'),
        ("Prepare data", step3_prepare_data, 'step3'),
        ("Choose model", step4_choose_model, 'step4'),
        ("Setup model", step5_setup_model, 'step5'),
        ("Configure training", step6_configure_training, 'step6'),
    ]

    for name, func, key in steps:
        if steps_completed.get(key, False):
            print(f"\n{name} - already done, skipping...")
            continue
        print(f"\n{'='*60}")
        print(f"RUNNING: {name}")
        print(f"{'='*60}")
        try:
            if not func():
                print(f"{name} FAILED!")
                return False
            print(f"{name} completed!")
        except Exception as e:
            print(f"Error in {name}: {e}")
            return False

    print("\n" + "="*60)
    print("ESSENTIAL STEPS COMPLETED!")
    print("="*60)
    return True

# ========== MAIN MENU ==========
def show_main_menu():
    print("\n" + "="*60)
    print("MAIN MENU")
    print("="*60)
    print("1. Start training")
    print("2. Load model")
    print("3. Create chatbot")
    print("4. Test with CSV")
    print("5. Start chatting")
    print("6. Save to Drive")
    print("7. Test accuracy")
    print("8. Restart")
    print("9. Exit")
    print("="*60)

    while True:
        choice = input("\nChoice (1-9): ").strip()
        if choice == "1" and steps_completed.get('step6'):
            step7_start_training()
        elif choice == "2":
            step8_load_model()
        elif choice == "3" and steps_completed.get('step8'):
            step9_create_chatbot()
        elif choice == "4":
            step10_test_with_csv()
        elif choice == "5" and steps_completed.get('step9'):
            step11_start_chatting()
        elif choice == "6":
            step12_save_to_drive()
        elif choice == "7":
            step13_test_accuracy()
        elif choice == "8":
            global df, question_cols, answer_cols, model_name, model_type
            global tokenizer, model, chatbot, csv_filename, trainer
            global fine_tuned_model, fine_tuned_tokenizer
            global train_dataset, val_dataset, tokenized_train, tokenized_val
            df = question_cols = answer_cols = model_name = model_type = None
            tokenizer = model = chatbot = csv_filename = trainer = None
            fine_tuned_model = fine_tuned_tokenizer = None
            train_dataset = val_dataset = tokenized_train = tokenized_val = None
            question_cols = []
            answer_cols = []
            for k in steps_completed:
                steps_completed[k] = False
            print("\nReset complete!")
            return "restart"
        elif choice == "9":
            return "exit"
        else:
            print("Invalid or prerequisites not met.")

def start_program():
    print("\n" + "="*60)
    print("WELCOME TO AI CHATBOT TRAINER")
    print("="*60)

    while True:
        print("\n" + "="*60)
        print("MAIN MENU")
        print("="*60)
        print("1. Run essential steps (1-6)")
        print("2. Full pipeline (train, load, chat)")
        print("3. Test with CSV")
        print("4. Start chatting")
        print("5. Save to Drive")
        print("6. Test accuracy")
        print("7. Exit")
        print("="*60)

        choice = input("\nChoice (1-7): ").strip()

        if choice == "1":
            if run_essential_steps():
                while True:
                    result = show_main_menu()
                    if result in ["exit", "restart"]:
                        break
        elif choice == "2":
            if not run_essential_steps():
                continue
            for name, func, key in [("Train", step7_start_training, 'step7'),
                                     ("Load", step8_load_model, 'step8'),
                                     ("Chatbot", step9_create_chatbot, 'step9')]:
                if steps_completed.get(key, False):
                    continue
                print(f"\n{'='*60}\nRunning: {name}\n{'='*60}")
                if not func():
                    break
            print("\nPipeline complete!")
        elif choice == "3":
            step10_test_with_csv()
        elif choice == "4":
            step11_start_chatting()
        elif choice == "5":
            step12_save_to_drive()
        elif choice == "6":
            step13_test_accuracy()
        elif choice == "7":
            print("Goodbye!")
            break

if __name__ == "__main__":
    start_program()

AI CHATBOT TRAINER - MULTI-MODEL SUPPORT
Supported model types:
  • Causal LM (decoder-only): DialoGPT, GPT-2, LLaMA, etc.
  • Seq2Seq (encoder-decoder): T5, Flan-T5, BART, etc.

WELCOME TO AI CHATBOT TRAINER

MAIN MENU
1. Run essential steps (1-6)
2. Full pipeline (train, load, chat)
3. Test with CSV
4. Start chatting
5. Save to Drive
6. Test accuracy
7. Exit

RUNNING ESSENTIAL STEPS

RUNNING: Install libraries

STEP 1: INSTALLING REQUIRED LIBRARIES
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 1.8 MB/s eta 0:00:00
Libraries installed successfully!
Install libraries completed!

RUNNING: Upload CSV

STEP 2: UPLOAD TRAINING CSV

Enter the path to your training CSV or type 'upload':
Loaded: /content/Test-Dataset-1.csv
Rows: 40, Columns: 7

Columns:
   1. Title
   2. Text
   3. Date
   4. _temporal_priority
   5. _source_priority
   6. _overall_priority
   7. _source

No QUESTION columns selected yet.

Current QUESTION columns selected:


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Training complete! Saved to: ./my-fine-tuned-chatbot

Running: Load

STEP 8: LOAD FINE-TUNED MODEL
Loading SEQ2SEQ model...


Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Model loaded on: cpu
Model dtype: torch.float32

Running: Chatbot

STEP 9: CREATE CHATBOT
   Chatbot type: SEQ2SEQ
   Chatbot created successfully!

Pipeline complete!

MAIN MENU
1. Run essential steps (1-6)
2. Full pipeline (train, load, chat)
3. Test with CSV
4. Start chatting
5. Save to Drive
6. Test accuracy
7. Exit

CHATBOT READY! (SEQ2SEQ)
Device: cpu
Commands: 'quit', 'clear', 'test'
Chatbot: CV (1995 experimental congenic episterium-free mebalesumycin virus of MDA
Chatbot: BSE' "Nitktbsh, U.Z".1 eDQ 285 "Nou katvyara se maatamnit dekatychmisklar.teztis-bişmetinim alwd kanadand unlünanlaznánr, provdajiran amik izinler." undk, 3 üset ve bukikalal kurogázidan para mekaldrymenunizdolaru analin-B.Q/C9
Chatbot: the law which inciteth one by doing something can mean lifesaving only on land and not for free-man’ to move through an uncertain land situation...I could have lost your lives in what my colleagues want


Chatbot: Goodbye!

MAIN MENU
1. Run essential steps (1-6)
2. Full pipeli